# GPLFR quickstart

A small, notebook-first introduction to GPLFR. This notebook fits a small GPLFR model and compares held-out predictions against the latent signal.

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch

from gplfr import GPLFR, create_synthetic_data

## Generate a small structured dataset

The synthetic generator produces a predictable signal plus spatially correlated
nuisance fields. We fit on the noisy observations and inspect held-out signal
recovery.

In [ ]:
data = create_synthetic_data(N=64, Dx=2, H=6, W=6, D_sig=3, ell=1.0, sigma_nuis=0.3, sigma_eps=0.01, seed=0)
split = 48

X_train, Y_train = data["X"][:split], data["Y"][:split]
X_test, Y_sig_test = data["X"][split:], data["Y_sig"][split:]
H, W = int(data["H"]), int(data["W"])

X_t = torch.from_numpy(X_train)
Y_t = torch.from_numpy(Y_train)
print(f"train X: {tuple(X_t.shape)}  train Y: {tuple(Y_t.shape)}  test: {len(X_test)}")

train X: (48, 2)  train Y: (48, 36)  test: 16


## Fit GPLFR

This keeps the settings intentionally small and notebook-friendly. We only pass
a few real overrides beyond the defaults.

In [ ]:
model = GPLFR(
    latent_dim=3,
    kernel="rbf",
    lengthscale_grouping="per_latent",
    amplitude_grouping="fixed",
    amplitude=1.0,
    device="auto",
)

fit_result = model.fit(
    X_t,
    Y_t,
    num_steps=200,
    verbose=False,
    seed=0,
)

print(f"final ELBO loss: {fit_result.final_loss:.4f}")

## Predict and inspect


In [ ]:
Y_pred = model.predict(torch.from_numpy(X_test)).reshape(-1, H, W)
Y_sig = Y_sig_test.reshape(-1, H, W)
print(f"predictions shape: {Y_pred.shape}")

fig, axes = plt.subplots(2, 2, figsize=(8, 7), constrained_layout=True)
for row, idx in enumerate((0, 1)):
    for col, (image, title) in enumerate(((Y_sig[idx], f"signal[{idx}]"), (Y_pred[idx], f"prediction[{idx}]"))):
        ax = axes[row, col]
        im = ax.imshow(image, cmap="viridis")
        ax.set_title(title)
        fig.colorbar(im, ax=ax)
plt.show()

## Next steps

- `model.py` for the GPLFR fit and predict API
- `synthetic.py` for the structured toy data generator
- the frozen reproduction zip for the exact paper artifact